# Koddan Kasa · Levha 11 · 12 eklem, iki sayı: 20 ve 0,5

Videodaki deney: **Unitree Go2** robot köpek yerde, bacakları katlı yatıyor. 12 ekleminin her birine aynı basit kuralı veriyoruz, her 2 ms'de bir:

```python
tau = Kp * (q_hedef - q) - Kd * q_hiz
```

- `q_hedef - q`: eklem hedef açıdan ne kadar uzakta (fark). **Kp** bu farkı torka çeviren katsayı: yay gibi.
- `q_hiz`: eklem ne hızla dönüyor. **Kd** bu hızı frenleyen katsayı: amortisör gibi.

**20 ve 0,5 nereden?** Unitree'nin açık eğitim kodu [`unitree_rl_gym` → `legged_gym/envs/go2/go2_config.py`](https://github.com/unitreerobotics/unitree_rl_gym/blob/main/legged_gym/envs/go2/go2_config.py): `control_type = 'P'`, `stiffness = {'joint': 20.}` (Kp, N·m/rad), `damping = {'joint': 0.5}` (Kd, N·m·s/rad). Hedef açılar da aynı dosyanın `default_joint_angles` sözlüğü (kalça ±0,1 · uyluk ön 0,8 / arka 1,0 · diz −1,5 rad).

**Dikkat, üç not:**
1. 20 / 0,5 **eğitim simülasyonunun** kazançları. Orada bu PD'nin üstünde bir yapay zekâ politikası çalışıyor ve hedef açıyı her adımda kaydırıyor (`default_joint_angles`, kodda "action = 0 iken hedef" diye geçiyor). Bu defterde politika yok: sadece çıplak PD.
2. Unitree'nin **gerçek** Go2 için yazdığı düşük seviye kalkma örneği farklı sayılar kullanıyor: **Kp 60, Kd 5** ([`unitree_sdk2/example/go2/go2_stand_example.cpp`](https://github.com/unitreerobotics/unitree_sdk2/blob/main/example/go2/go2_stand_example.cpp)).
3. Robot modeli [MuJoCo Menagerie `unitree_go2`](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_go2) (BSD-3 © Unitree Robotics, Google DeepMind'ın paketi). Bu modelde eklemlerin kendi **pasif sönümü 2** (N·m·s/rad), sürtünmesi 0,2. Yani eğitim ortamıyla birebir aynı değil: buradaki salınımlar eğitim simülasyonundakinden daha yumuşak. Bizim sayılar bu modelle ölçüldü.

Bizim sonuçlar (MuJoCo 3.17.0, Menagerie commit `cda5236`): 20 / 0,5 → gövde **25,2 cm**'de duruyor, tepe 25,6 cm, aşma %3,0 · Kp 5 → 11,1 cm, kalkamıyor · Kp 80 → tepe **50,8 cm**, aşma %104,5 · Kd 0 → aşma %11,0 · +5 kg yük → 21,8 cm.

Sırayla çalıştır (Çalışma zamanı → Tümünü çalıştır). Sonra **BURAYI DEĞİŞTİR** hücresinde iki sayıyı değiştir, tekrar çalıştır.

## 1 · Kurulum (MuJoCo + sadece Go2 modeli)

In [ ]:
!pip -q install mujoco mediapy
# Birebir aynı sayılar için: !pip -q install mujoco==3.17.0 mediapy
![ -d mujoco_menagerie ] || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie
!cd mujoco_menagerie && git sparse-checkout set unitree_go2 && ls unitree_go2

## 2 · Deney: yatıştan kalkış

Protokol (`src/v6/kk11v3/experiment.py` ile aynı):
- Bacaklar katlı (uyluk 1,25, diz −2,7 rad), gövde 12 cm'den bırakılıyor; 0,5 sn bu pozda tutulup yere oturtuluyor.
- Sonra 3 sn boyunca 12 ekleme `tau = Kp*(q* - q) - Kd*q'` veriliyor, tork motorun kendi sınırında kırpılıyor (kalça/uyluk ±23,7, diz ±45,43 N·m).
- Ölçümler gövde (base) yüksekliği üzerinden:
  - **son** = son 0,5 sn'nin ortalaması (2,5–3 sn),
  - **tepe** = en yüksek değer,
  - **aşma** = (tepe − son) / (son − başlangıç) × 100. Hedefe göre değil, robotun kendi oturduğu yüksekliğe göre.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")  # Colab'da ekransız çizim
import mujoco
import numpy as np

XML = "mujoco_menagerie/unitree_go2/scene.xml"

# unitree_rl_gym go2_config.py default_joint_angles, sıra: FL FR RL RR × (hip, thigh, calf)
STAND = np.array([0.1, 0.8, -1.5, -0.1, 0.8, -1.5, 0.1, 1.0, -1.5, -0.1, 1.0, -1.5])
LIE = np.array([0.0, 1.25, -2.7] * 4)  # katlı bacaklar, karın yere yakın
SIM_SEC = 3.0


def run(kp, kd, load=0.0, renderer=None, fps=30):
    m = mujoco.MjModel.from_xml_path(XML)
    if load:
        m.body_mass[m.body("base").id] += load
    d = mujoco.MjData(m)
    d.qpos[7:] = LIE
    d.qpos[2] = 0.12
    mujoco.mj_forward(m, d)
    lo, hi = m.actuator_ctrlrange[:, 0], m.actuator_ctrlrange[:, 1]
    # önce katlı pozu yerde oturt (LIE'yi tut)
    for _ in range(int(0.5 / m.opt.timestep)):
        d.ctrl[:] = np.clip(40 * (LIE - d.qpos[7:]) - 1.0 * d.qvel[6:], lo, hi)
        mujoco.mj_step(m, d)
    t0 = d.time
    log, frames = [], []
    every = max(1, int(round(1 / 120 / m.opt.timestep)))
    every_frame = max(1, int(round(1 / fps / m.opt.timestep)))
    cam = mujoco.MjvCamera()
    cam.lookat[:] = [d.qpos[0], d.qpos[1], 0.2]
    cam.distance, cam.azimuth, cam.elevation = 1.3, 130, -15
    i = 0
    while d.time - t0 < SIM_SEC:
        tau = kp * (STAND - d.qpos[7:]) - kd * d.qvel[6:]   # <- bütün denetleyici bu satır
        d.ctrl[:] = np.clip(tau, lo, hi)
        mujoco.mj_step(m, d)
        if i % every == 0:
            log.append([d.time - t0, float(d.qpos[2]), float(np.abs(d.ctrl).max() / hi.max()), *d.qpos[7:]])
        if renderer is not None and i % every_frame == 0:
            renderer.update_scene(d, camera=cam)
            frames.append(renderer.render())
        i += 1
    a = np.round(np.array(log), 4)
    t, h = a[:, 0], a[:, 1]
    start, final, peak = float(h[0]), float(h[t > SIM_SEC - 0.5].mean()), float(h.max())
    over = max(0.0, (peak - final) / (final - start) * 100) if final > start + 0.01 else 0.0
    qerr = np.abs(STAND - a[-1, 3:]).max()
    s = dict(Kp=kp, Kd=kd, load_kg=load, start_h=round(start, 3), final_h=round(final, 3), peak_h=round(peak, 3),
             overshoot_pct=round(over, 1), joint_err_deg=round(float(np.degrees(qerr)), 2))
    return s, a, frames


print("mujoco", mujoco.__version__)

## 3 · ✏️ BURAYI DEĞİŞTİR

Unitree'nin eğitim sayıları: `KP = 20`, `KD = 0.5`. Dene: `KP = 5` (kalkabiliyor mu?), `KP = 80` (ne oluyor?), `KD = 0` (fren yokken?), `KP = 60, KD = 5` (gerçek köpek örneğinin sayıları). İstersen `LOAD_KG = 5` ile sırtına yük koy.

In [ ]:
# ===================== BURAYI DEĞİŞTİR =====================
KP = 20      # yay: farkı torka çeviren sayı   [N·m/rad]
KD = 0.5     # fren: hızı yavaşlatan sayı      [N·m·s/rad]
LOAD_KG = 0  # sırttaki ek yük [kg]
# ===========================================================

s, a, _ = run(KP, KD, load=LOAD_KG)
print(f"Kp={KP}  Kd={KD}  yük={LOAD_KG} kg")
print(f"başlangıç : {s['start_h']*100:.1f} cm")
print(f"son       : {s['final_h']*100:.1f} cm   (2,5–3 sn ortalaması)")
print(f"tepe      : {s['peak_h']*100:.1f} cm")
print(f"aşma      : %{s['overshoot_pct']:.1f}")
print(f"en çok geride kalan eklem: {s['joint_err_deg']:.1f}°")

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.2))
plt.plot(a[:, 0], a[:, 1] * 100, lw=2, label=f"Kp {KP} · Kd {KD}")
plt.axhline(s["final_h"] * 100, ls="--", lw=1, color="gray", label="son yükseklik")
plt.xlabel("zaman [sn]"); plt.ylabel("gövde yüksekliği [cm]")
plt.title("Go2 yatıştan kalkış · tau = Kp·(q* − q) − Kd·q'")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

### Videosu (480×360, 30 fps)

In [ ]:
import mediapy as media
renderer = mujoco.Renderer(mujoco.MjModel.from_xml_path(XML), height=360, width=480)
_, _, frames = run(KP, KD, load=LOAD_KG, renderer=renderer, fps=30)
renderer.close()
media.show_video(frames, fps=30)

## 4 · Tablo: bizim koşular

Videodaki tablonun tamamı. Her satırı yukarıdaki kuralla yeniden koşuyor ve bizim kaydettiğimiz sayılarla (`go2_pd.json`, MuJoCo 3.17.0) karşılaştırıyor. Farklı bir MuJoCo ya da Menagerie sürümünde son basamaklar biraz kayabilir.

In [ ]:
CASES = {"unitree (20 / 0,5)": (20, 0.5, 0), "kp5": (5, 0.5, 0), "kp10": (10, 0.5, 0), "kp40": (40, 0.5, 0),
         "kp80": (80, 0.5, 0), "kd0": (20, 0.0, 0), "kd2": (20, 2.0, 0), "+5 kg yük": (20, 0.5, 5.0)}
# bizim sonuçlar: public/v6/kk-11-v3/data/go2_pd.json  (son_h, tepe_h, aşma_%)
OURS = {"unitree (20 / 0,5)": (0.252, 0.256, 3.0), "kp5": (0.111, 0.111, 0.8), "kp10": (0.196, 0.197, 0.1),
        "kp40": (0.291, 0.36, 36.4), "kp80": (0.299, 0.508, 104.5), "kd0": (0.247, 0.263, 11.0),
        "kd2": (0.252, 0.253, 1.2), "+5 kg yük": (0.218, 0.227, 7.4)}

results = {}
print(f"{'koşu':<20}{'Kp':>5}{'Kd':>6}{'son cm':>9}{'tepe cm':>9}{'aşma %':>9}   bizimkiyle")
for name, (kp, kd, load) in CASES.items():
    s, a, _ = run(kp, kd, load=load)
    results[name] = (s, a)
    got = (s["final_h"], s["peak_h"], s["overshoot_pct"])
    same = "✓ aynı" if got == OURS[name] else f"≠ bizim: {OURS[name]}"
    print(f"{name:<20}{kp:>5}{kd:>6}{s['final_h']*100:>9.1f}{s['peak_h']*100:>9.1f}{s['overshoot_pct']:>9.1f}   {same}")

plt.figure(figsize=(8, 3.6))
for name in ["kp5", "kp10", "unitree (20 / 0,5)", "kp40", "kp80"]:
    s, a = results[name]
    plt.plot(a[:, 0], a[:, 1] * 100, lw=2 if name.startswith("unitree") else 1.2, label=name)
plt.xlabel("zaman [sn]"); plt.ylabel("gövde yüksekliği [cm]"); plt.title("Sadece Kp değişince")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

## 5 · Hedef duruş aslında kaç cm? (7 santimlik açık)

Hedef açılar tam tutulsaydı gövde ne kadar yüksekte olurdu? Robotu havada, gövdesi düz, eklemleri tam `default_joint_angles` açısında koyuyoruz ve ayak küresinin altından gövdeye olan dikey mesafeyi ölçüyoruz (fizik yok, sadece geometri). Ön bacaklar biraz daha uzun kalıyor; gövde düzken yere önce onlar değiyor.

In [ ]:
m = mujoco.MjModel.from_xml_path(XML)
d = mujoco.MjData(m)
d.qpos[:3] = [0, 0, 1.0]          # havada
d.qpos[3:7] = [1, 0, 0, 0]        # gövde düz
d.qpos[7:] = STAND
mujoco.mj_kinematics(m, d)
leg_h = {}
for foot in ["FL", "FR", "RL", "RR"]:
    g = m.geom(foot).id
    bottom = d.geom_xpos[g][2] - m.geom_size[g][0]   # ayak küresinin en alt noktası
    leg_h[foot] = d.qpos[2] - bottom
    print(f"{foot}: {leg_h[foot]*100:.2f} cm")
kin_h = max(leg_h.values())
final_20 = results["unitree (20 / 0,5)"][0]["final_h"] if "results" in globals() else run(20, 0.5)[0]["final_h"]
print(f"\nhedef duruşun kinematik yüksekliği : {kin_h*100:.2f} cm")
print(f"20 / 0,5 ile ulaşılan (son)        : {final_20*100:.2f} cm")
print(f"açık                               : {(kin_h - final_20)*100:.1f} cm")

## 6 · Soru: Unitree gerçek köpekte neden 60 / 5 kullanıyor?

20 / 0,5 ile gövde hedef duruşun ~7 cm altında kalıyor; Kp'yi büyütünce sarkma azalıyor ama robot fırlıyor. Gerçek Go2 örneğindeki 60 / 5'i **BURAYI DEĞİŞTİR** hücresinde dene, sonra şu ipuçlarını düşün:

- [`go2_stand_example.cpp`](https://github.com/unitreerobotics/unitree_sdk2/blob/main/example/go2/go2_stand_example.cpp)'ye bak: hedef açılar robota **bir anda** mı veriliyor, yoksa başlangıç pozundan hedefe doğru adım adım mı kaydırılıyor? (Bizim deneyde hedef t = 0'da tek seferde veriliyor.) Kaydırılırsa büyük Kp neden daha az fırlatır?
- Eğitim simülasyonunda PD'nin üstünde, hedefi her adımda düzelten bir **yapay zekâ politikası** var; sarkmayı o kapatabilir. Gerçek-köpek örneğinde böyle bir düzelten yok: sarkmayı kim kapatacak?
- Kd 5, Kp 60'a oranla 0,5 / 20'den daha mı büyük, daha mı küçük? Yukarıdaki `kd0` / `kd2` satırları sana ne söylüyor?
- Bu model gerçek köpek değil (pasif sönüm 2, motorlar ideal tork kaynağı). Buradaki sonuç gerçek robotta da aynı çıkar mı? Ne zaman güvenebiliriz?

Cevabı yorumlara yaz.